# 16.6 Adopting Types in an Existing Codebase

**Prerequisites:** 16.1–16.5, 15.6 Testing in Practice  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 Why `--strict` on day one is the way to fail
- `--check-untyped-defs` — free bug-finding before you annotate anything
- Configuring mypy in `pyproject.toml`
- **Per-module strictness** — a ratchet you can actually tighten
- The order to annotate in, and why it is signatures first
- Wiring the checker into CI beside the tests (**15.6**)
- What to do about the errors you cannot fix today
- 🔴 Where types genuinely do not pay for themselves
- Interview questions

---

## The situation

You have 40,000 lines with no annotations. Everything in **16.1–16.5** assumed a clean slate.
This notebook is about the realistic case, and it has one governing rule:

> 🔴 **Never turn on `--strict` for a legacy codebase.**
> You get thousands of errors, none of them prioritised, and the team concludes that typing is
> not worth it. That conclusion is wrong, but the experience that produced it was real.

Adoption works as a **ratchet**: make a small area strict, prevent backsliding, widen. The rest
of this notebook is the mechanics of that.

Below is a small "legacy" project to work on — three modules, no annotations, and one real
bug.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py166_"))
PROJECT = WORK / "legacy_app"


def write(name, source, root=None):
    path = (root or PROJECT) / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return name


def mypy(target, *flags, root=None):
    """Run mypy against the demo project and return its report."""
    done = subprocess.run(
        [sys.executable, "-m", "mypy", target,
         "--cache-dir", str(WORK / ".mypy_cache"),
         "--no-color-output", "--no-error-summary", *flags],
        cwd=root or PROJECT, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=300)
    report = (done.stdout + done.stderr).strip() or "(mypy found nothing to report)"
    return (f"$ mypy {target} {' '.join(flags)}".rstrip() + "\n" + "-" * 68 + "\n"
            + report + "\n" + "-" * 68 + f"\nexit code: {done.returncode}")


print("scratch:", WORK)

In [ ]:
write("app/__init__.py", "")

write("app/retry.py", r"""
    def retry_delay(attempt, base=1.0, ceiling=30.0):
        delay = base
        for _ in range(attempt):
            delay = delay * 2
        return min(delay, ceiling)


    def is_retryable(status, attempts, max_attempts=3):
        return status in (500, 502, 503) and attempts < max_attempts
""")

write("app/spool.py", r"""
    def load_budget(raw):
        return int(raw)


    def summarise(rows):
        # 🔴 a real bug: `total` may be None, and None + int explodes
        total = None
        for row in rows:
            value = load_budget(row)
            if total is None:
                total = value
            else:
                total = total + value
        return total / len(rows)
""")

write("app/report.py", r"""
    from app.spool import summarise


    def render(rows):
        average = summarise(rows)
        return "average budget: " + average
""")

print("legacy project created:")
for path in sorted(PROJECT.rglob("*.py")):
    print("   ", path.relative_to(PROJECT))

## Step 1 — `--check-untyped-defs`, before annotating anything

The single best first move. It tells mypy to look **inside** unannotated function bodies, which
it normally skips (**16.1**). You annotate nothing and get real bug-finding immediately.

🔴 It is **not** part of `--strict`, which is why most people never discover it.

In [ ]:
print(mypy("app"))
print()
print(mypy("app", "--check-untyped-defs"))

The default run found **nothing** — every function is unannotated, so there is
nothing to contradict.

`--check-untyped-defs` found a real defect, with **zero annotations added**:

```
app/spool.py:14: error: Unsupported operand types for / ("None" and "int")
app/spool.py:14: note: Left operand is of type "Any | None"
```

`summarise` initialises `total = None`, and if `rows` is empty the loop never runs — so
`total / len(rows)` is `None / 0`, which is two bugs stacked on one line. The checker traced
`total` through the branch and worked out it could still be `None` at the end.

That is the argument to make to a sceptical team: **run one flag, find real bugs, annotate
nothing.**

## Step 2 — a configuration file

Put settings in `pyproject.toml` so everyone — editor, CLI, CI — behaves identically
(**15.6** made the same argument for pytest).

```toml
[tool.mypy]
python_version = "3.12"
files = ["app", "tests"]

# Sensible for any codebase, strict or not:
check_untyped_defs = true       # look inside unannotated bodies
warn_unused_ignores = true      # stale `type: ignore` becomes an error (16.1)
warn_redundant_casts = true     # a cast that changes nothing (16.5)
warn_unreachable = true
show_error_codes = true

# The ratchet: strict everywhere, with named exceptions that shrink over time.
strict = true

[[tool.mypy.overrides]]
module = ["app.legacy.*", "app.spool"]
disallow_untyped_defs = false

[[tool.mypy.overrides]]
module = ["nonexistent_vendor_lib.*"]
ignore_missing_imports = true
```

🔴 **The direction matters.** Turn `strict` on globally and list the exemptions, rather than
turning strictness on module by module. The exemption list is then a **visible, shrinking
to-do list** — and a new module is strict by default instead of silently unchecked.

In [ ]:
write("pyproject.toml", r"""
    [tool.mypy]
    check_untyped_defs = true
    strict = true
""")
print("### strict everywhere, no exemptions")
print(mypy("app"))

Every unannotated function is now an error. On a real codebase that is the
thousands-of-errors wall.

Now the same project with **one module exempted**.

In [ ]:
write("pyproject.toml", r"""
    [tool.mypy]
    check_untyped_defs = true
    strict = true

    # Shrinking to-do list: each entry is a module still awaiting annotation.
    [[tool.mypy.overrides]]
    module = ["app.spool", "app.report"]
    disallow_untyped_defs = false
    disallow_untyped_calls = false
""")
print("### strict, with app.spool and app.report exempted")
print(mypy("app"))

`app/retry.py` is now held to `--strict` while the two exempted modules are
not — and the real bug from step 1 is *still* reported, because `check_untyped_defs` is on
globally.

That is the ratchet: **the errors you see are the ones you have chosen to fix.** Delete an entry
from the overrides list when a module is done, and it can never regress.

## Step 3 — the order to annotate in

Not all annotations are worth the same. In rough order of value per minute:

| Order | What | Why |
|---|---|---|
| 1 | **Function signatures** at module boundaries | 🔴 stops `Any` leaking outward (**16.1**) |
| 2 | **Return types** everywhere | an unannotated return is `Any` for every caller |
| 3 | Data structures — `TypedDict`, `dataclass` (**16.2**) | one definition types dozens of call sites |
| 4 | Local variables | 🔴 rarely — inference handles almost all of it |

🔴 **Do not annotate local variables by reflex.** `count: int = 0` adds noise; the checker
already knows. Annotate a local only when inference genuinely cannot help — an empty container
(`items: list[str] = []`) or a deliberately widened type.

Watch what one signature does to a caller in a *different* module.

In [ ]:
write("app/spool.py", r"""
    from collections.abc import Sequence


    def load_budget(raw: str) -> int:
        return int(raw)


    def summarise(rows: Sequence[str]) -> float:
        total = 0
        for row in rows:
            total += load_budget(row)
        return total / len(rows)
""")

write("pyproject.toml", r"""
    [tool.mypy]
    check_untyped_defs = true
    strict = true

    [[tool.mypy.overrides]]
    module = ["app.report"]
    disallow_untyped_defs = false
    disallow_untyped_calls = false
""")

print("### app.spool annotated; app.report still exempt")
print(mypy("app"))

One module annotated, and the error in `app/report.py` — a *different,
unannotated* file — is now reported precisely: `Unsupported operand types for + ("str" and
"float")`.

🔴 **That is the compounding effect.** Annotating a module improves checking in every module
that *calls* it, whether or not those are annotated themselves. It is why boundaries and
signatures come first.

Also note `summarise` is now genuinely fixed: starting `total` at `0` rather than `None`
removed the `None` handling entirely and made the code shorter.

## Step 4 — errors you cannot fix today

Three tools, in descending order of preference:

| Tool | Scope | Use for |
|---|---|---|
| `# type: ignore[code]` | one line | a genuine checker limitation, with a comment |
| per-module override | one module | a file awaiting its turn |
| `follow_imports = "skip"` | a whole subtree | 🔴 last resort — it hides everything |

🔴 **Always the narrowest that works**, always with the error code (**16.1**), and always with
`warn_unused_ignores = true` so they cannot outlive their reason.

A useful trick for a big migration: `mypy --strict app | grep -oE '\[[a-z-]+\]' | sort | uniq -c`
gives you a histogram of error codes. Fix the largest category first — it is usually one
mechanical change repeated hundreds of times.

## Step 5 — CI

Types earn their keep only if a failure stops the build. This is **15.6**'s pipeline with the
type step in place:

```yaml
      - run: ruff check .                          # lint            (17)
      - run: mypy                                  # types           (this folder)
      - run: |                                     # tests           (15)
          coverage run -m pytest -ra --strict-markers
          coverage report --fail-under=80
```

Two details that matter:

- **Pin the mypy version.** A new release finds new errors; that is good, but it should be a
  deliberate upgrade rather than a red build on an unrelated PR.
- **Run mypy on the *tests* too.** Test code is code, and typing it catches wrong mock
  signatures — the exact bug **15.5** showed `autospec` catching.

> A pre-commit hook gives faster feedback, but CI is what makes it binding.

## 🔴 Where types do not pay

Being honest about this is what makes the rest credible.

| Situation | Why |
|---|---|
| A 30-line script you will run twice | the annotation costs more than the bug would |
| Genuinely dynamic code | plugin registries, ORMs and `**kwargs` passthroughs fight the checker |
| Exploratory notebook work | you are discovering the shape, not documenting it |
| Chasing 100% strictness on a huge legacy tree | the last 5% costs more than the first 95% |
| Anything you would type `Any` to silence | 🔴 an `Any` there is worse than no annotation — it *looks* checked |

And the limit from **16.1**, worth restating because it is the one people forget:

> **A green type check says nothing about whether your logic is right.** The `--strict`-clean
> file in 16.1 had three real bugs. Types and tests are complementary and neither replaces the
> other.

## Interview Questions

1. **What does a type annotation do at runtime?** *(nothing — 16.1)*
2. **Why is an unannotated function dangerous in a typed codebase?** *(it returns `Any`, which
   disables checking downstream — 16.1)*
3. **`Any` vs `object` — when would you use each?** *(16.1)*
4. **Why is `list[BuildJob]` not a `list[Job]`?** *(invariance; the `sabotage` demonstration —
   16.3)*
5. **When would you take `Sequence` rather than `list` as a parameter?** *(covariance; accept
   widely, return specifically — 16.3)*
6. **`Protocol` or `ABC`?** *(structural vs nominal; can you edit the implementers? — 16.4)*
7. **What does `isinstance` against a `runtime_checkable` protocol actually check?**
   *(member presence only — not signatures, not types — 16.4)*
8. **Why return `Self` instead of the class name?** *(subclasses; fluent chains — 16.4)*
9. **What does `cast` do at runtime?** *(nothing; it is an unchecked assertion that can crash a
   line later — 16.5)*
10. **How do you type a function that returns `str` with a default and `str | None` without
    one?** *(`@overload` — 16.4)*
11. **You are asked to add types to a 40,000-line codebase. What is your first move?**
    *(`--check-untyped-defs`, not `--strict` — this notebook)*
12. **How do you stop a partially-typed codebase from regressing?** *(strict globally with a
    shrinking override list, enforced in CI)*
13. **Where would you argue *against* adding types?** *(the table above — an honest answer is
    better than a dogmatic one)*

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Turning on `--strict` first.** Thousands of unprioritised errors, and a team that concludes typing is not worth it.
2. **Never trying `--check-untyped-defs`.** It is not in `--strict`, needs no annotations, and finds real bugs on day one.
3. **Enabling strictness module by module instead of globally with exemptions.** New modules then start unchecked, and there is no visible to-do list.
4. **Annotating local variables by reflex.** Inference handles almost all of them; the noise makes real annotations harder to see.
5. 🔴 **Annotating a parameter `Any` to make an error go away.** It looks checked and is not — worse than leaving it bare.
6. **Leaving mypy out of CI.** A checker nobody enforces is a checker nobody runs.
7. **Not pinning the mypy version.** A release lands and an unrelated PR goes red.
8. **Skipping the tests when type-checking.** Test code has bugs too, and wrong mock signatures are exactly what types catch (**15.5**).
9. **Chasing 100%.** The last few percent of a legacy tree costs more than everything before it.
10. **Believing a green check means correct.** The `--strict`-clean file in **16.1** had three real bugs.

## Best Practices

- Start with `--check-untyped-defs` and fix what it finds, before annotating anything.
- Put configuration in `pyproject.toml` so editor, CLI and CI agree.
- Set `strict = true` globally and keep a **shrinking list of per-module exemptions**.
- Annotate signatures at module boundaries first, return types next, data structures third, locals almost never.
- Turn on `warn_unused_ignores` so silenced errors cannot outlive their reason.
- Group errors by code and fix the largest category first — it is usually one repeated fix.
- Run mypy in CI next to pytest, with a pinned version, over source **and** tests.
- Delete an override the day its module is clean; that is the ratchet clicking.
- Be honest about where types do not pay — it is what makes the case credible elsewhere.

## Practice Exercises

Try these before moving on.

1. Run `mypy --check-untyped-defs` on a project of your own that has no annotations. How many real bugs did it find, and how long did it take?
2. Write a `pyproject.toml` with `strict = true` and one per-module exemption. Prove the exemption changes the result, then delete it and fix the module.
3. 🔴 Annotate one module's signatures and count how many *new* errors appear in modules you did not touch. That number is the compounding effect.
4. Produce an error-code histogram with `mypy --strict . | grep -oE '\[[a-z-]+\]' | sort | uniq -c | sort -rn`. What is your largest category, and is it one mechanical fix?
5. Add mypy to the CI workflow from **15.6**, over both `src` and `tests`. Make it fail once for the right reason.
6. Take a **14 Data Structure and Algorithm** notebook, extract the code into a module, and annotate it with generics (**16.3**). Does `Stack[T]` read better than `Stack`?
7. 🔴 Find a piece of code in your own work where you would argue *against* adding types. Write the argument down in two sentences — could you defend it in review?
8. **Capstone:** take the untyped legacy project from this notebook, annotate it fully, get it clean under `--strict` with **no** overrides, and add pytest tests (**15.1**) for the bug that `--check-untyped-defs` found.

---

## Version notes

| Version | Change |
|---|---|
| **mypy 1.5+** | `[[tool.mypy.overrides]]` in `pyproject.toml` is the standard form; `mypy.ini` still works |
| **3.14** | PEP 649 lazy annotations — less need for `from __future__ import annotations` (**16.5**) |
| **3.12** | PEP 695 generics (**16.3**); `@override` for verified overriding |
| **3.11** | `Self` (**16.4**), `assert_never` (**16.2**) |
| **3.10** | `X | Y` unions (**16.2**), `ParamSpec` (**16.3**) |

> **Other checkers.** `pyright` (which powers Pylance in VS Code) is faster and often stricter;
> `ty` and `pyrefly` are newer entrants. They read the same annotations, so nothing in this
> folder is mypy-specific — only the flag names differ.

## 16 Type Hints and Static Typing — the folder

| Notebook | Covers |
|---|---|
| **16.1** | annotations are not enforced; `reveal_type`; `Any` vs `object`; `--strict`; what types miss |
| **16.2** | unions, narrowing, the truthiness trap, `Literal`, `assert_never`, `Final`, `TypedDict` |
| **16.3** | generics — PEP 695, bounds vs constraints, variance, `ParamSpec` |
| **16.4** | protocols, structural typing, `runtime_checkable`'s limits, `Self`, `@overload` |
| **16.5** | generators, async, context managers, `cast`, `TYPE_CHECKING`, third-party stubs |
| **16.6** | this notebook — adopting types in an existing codebase |

**The one-sentence version:** *types are a second opinion that reads every path, tests are the
opinion that knows what the answer should be, and you want both.*

## Related

- **4.5 Type Hints for Functions** — where the syntax was introduced
- **15.6 Testing in Practice** — the CI pipeline this slots into, and the same "green is not
  correct" lesson for coverage
- **15.5 Test Doubles** — why typing your tests catches wrong mock signatures
- **17 Tooling, Packaging and Environments** — `pyproject.toml`, `ruff`, `py.typed`, publishing
- **19 Capstone Projects** — where a typed, tested codebase finally pays off